# CS781 — Phase 2, Architecture 2 (Legal NER) — updated run

This is my **Architecture 2 partial fine-tuning** run. It downloads the encoder weights and BPE tokenizer from `P-261110044/Phase1-A2`. The decoder from Phase 1 is not needed for NER.

Changes kept from the debugging runs:
- Exact Phase-1 checkpoint reconstruction, without loading the custom model through `AutoModel`.
- Whitespace-aware token-to-gold alignment and a gold-span recovery check **before training**.
- 256-token windows, moving 192 tokens each time (64 tokens overlap).
- Overlapping window logits merged **before** BIO decoding, so duplicates are not turned into extra entities.
- Greedy and valid-BIO Viterbi decoding compared on the validation documents.
- Class-weighted loss (`O` weight 0.5), separate encoder and classifier learning rates, gradient accumulation.
- Up to 30 epochs. **A training-loss plateau is not a stopping condition.** The full 30 epochs run by default; best checkpoint is chosen using validation strict Macro-F1.
- Span checks, per-label F1, prediction ZIP, and optional upload of Phase-2 outputs to a **separate private** Hugging Face repo.

This is **one** Phase-2 method (partial/layer-wise fine-tuning). Other adaptation and distillation experiments remain separate. Run top to bottom on Kaggle with the `HF` secret and Internet enabled.


In [1]:
# Kaggle has PyTorch preinstalled. Do not install PEFT/torchao here:
# this custom PyTorch checkpoint is deliberately loaded without PEFT/AutoModel.
# Only the Hub, tokenizer and safetensors libraries are needed here.
%pip -q install -U "huggingface_hub>=0.31,<2" "transformers>=4.46,<5" "safetensors>=0.4.5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 1. Settings and Hugging Face authentication

In [2]:
import os, json, re, random, zipfile, gc, time, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download, login
from transformers import PreTrainedTokenizerFast
from safetensors.torch import load_file
from kaggle_secrets import UserSecretsClient

# Keep the split and training order reproducible.
SEED = 44
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Same secret I used in Phase 1.
HF_TOKEN = UserSecretsClient().get_secret('HF')
assert HF_TOKEN, 'Add a Kaggle secret named HF with read access to the Phase-1 repo.'
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)

PHASE1_REPO = 'P-261110044/Phase1-A2'
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
ARTIFACT_PREFIX = 'cs781_phase1_a2_encoderdecoder/legal-gpt2-final/'
OUT = Path('/kaggle/working/phase2_a2_partial_updated')
CACHE = OUT / 'phase1_download'
OUT.mkdir(parents=True, exist_ok=True)

# The original A2 notebook used 600 hidden units and 15 heads.
# Hidden width/layer counts are checked against the saved tensors below.
HEADS_OVERRIDE = None
UNFREEZE_LAST_N = 2
MAX_TOKENS = 256
WINDOW_STEP = 192                # 64 tokens are shared between neighbouring windows
BATCH_SIZE = 1
ACCUM_STEPS = 4
EPOCHS = 30

# Keep training through the requested epochs, even if loss goes up or stays flat.
# If I want patience-based stopping later, it uses validation F1, not training loss.
USE_EARLY_STOPPING = True
PATIENCE = 5

LR_ENCODER = 2e-5
LR_HEAD = 2e-4
O_WEIGHT = 0.5
WEIGHT_DECAY = 0.01
VAL_FRACTION = 0.10
MAX_TRAIN_DOCS = None
MAX_VAL_DOCS = None
MAX_TRAIN_WINDOWS_PER_DOC = None  # None = do not drop training windows
STOP_IF_ALIGNMENT_BAD = True
MIN_GOLD_RECOVERY = 0.80
AUTO_SELECT_DECODER = True
UPLOAD_PHASE2_CHECKPOINT = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Hugging Face:', api.whoami()['name'])
print('Device:', DEVICE)
print('Output:', OUT)


Hugging Face: P-261110044
Device: cuda
Output: /kaggle/working/phase2_a2_partial_updated


## 2. Download exact Phase-1 weights and tokenizer

Only two or three relatively small artifacts are necessary; optimizer, scheduler, `trainer_state.json`, and the full ZIP are **not** required for task fine-tuning.

In [3]:
# List the repo first, so a missing filename gives a clear error.
remote_files = set(api.list_repo_files(PHASE1_REPO, repo_type='model'))

# Download one file from Phase 1 into the local cache.
def grab(relative_path):
    if relative_path not in remote_files:
        raise FileNotFoundError(f'Missing in {PHASE1_REPO}: {relative_path}')
    return hf_hub_download(repo_id=PHASE1_REPO, filename=relative_path,
                           repo_type='model', token=HF_TOKEN,
                           local_dir=str(CACHE))

# Prefer the final exported weights. Use the latest available checkpoint only if needed.
weights_rel = ARTIFACT_PREFIX + 'model_state.pt'
if weights_rel in remote_files:
    weight_path = grab(weights_rel)
    print('Using final state dict:', weights_rel)
else:
    candidates = [x for x in remote_files if
                  x.startswith(ARTIFACT_PREFIX+'checkpoint-') and x.endswith('/model.safetensors')]
    if not candidates:
        raise FileNotFoundError('Neither final model_state.pt nor checkpoint model.safetensors was found.')
    weights_rel = max(candidates, key=lambda s: int(re.search(r'checkpoint-(\d+)',s).group(1)))
    weight_path = grab(weights_rel)
    print('Using fallback checkpoint:', weights_rel)

# Use exactly the BPE tokenizer that was saved with this model.
for name in ['tokenizer.json', 'tokenizer_config.json']:
    grab(ARTIFACT_PREFIX + name)

TOKENIZER_DIR = CACHE / ARTIFACT_PREFIX.rstrip('/')
tokenizer = PreTrainedTokenizerFast.from_pretrained(str(TOKENIZER_DIR))
assert tokenizer.is_fast, 'Fast tokenizer required for original-character offset mapping.'

# The final weights are a .pt file; Trainer checkpoints use safetensors.
if weight_path.endswith('.safetensors'):
    original_state = load_file(weight_path, device='cpu')
else:
    original_state = torch.load(weight_path, map_location='cpu', weights_only=True)
if isinstance(original_state, dict) and 'state_dict' in original_state:
    original_state = original_state['state_dict']
original_state = {k.removeprefix('module.'):v for k,v in original_state.items()}

# Read dimensions from the actual weight tensors to avoid size-mismatch errors.
vocab_size, d_model = original_state['token_embedding.weight'].shape
source_length, positional_width = original_state['source_position.weight'].shape
n_layers = 1 + max(int(m.group(1)) for key in original_state
                   if (m := re.match(r'encoder_blocks\.(\d+)\.', key)))
d_ff = original_state['encoder_blocks.0.feed_forward.0.weight'].shape[0]

# Head count is different: its value is not stored in the tensor dimensions.
# MHA number of heads cannot be deduced from state_dict shapes. Original A2 uses 15.
n_heads = HEADS_OVERRIDE if HEADS_OVERRIDE is not None else {600:15, 512:8}.get(d_model)
if n_heads is None:
    raise ValueError(f'Unknown heads for d_model={d_model}: set HEADS_OVERRIDE from training config.')
assert d_model % n_heads == 0 and positional_width == d_model
# The number of tokenizer IDs must match the rows in the saved embedding table.
assert len(tokenizer) == vocab_size, (f'Tokenizer has {len(tokenizer)} IDs, '
                                      f'but checkpoint has {vocab_size} embeddings')
assert MAX_TOKENS <= source_length
assert tokenizer.pad_token_id is not None, 'Phase-1 tokenizer must define <pad>.'

print({'vocab_size':vocab_size, 'd_model':d_model, 'n_encoder_layers':n_layers,
       'd_ff':d_ff, 'heads_selected_from_original_source':n_heads,
       'encoder_position_limit':source_length, 'tokenizer_size':len(tokenizer)})

cs781_phase1_a2_encoderdecoder/legal-gpt(…):   0%|          | 0.00/541M [00:00<?, ?B/s]

Using final state dict: cs781_phase1_a2_encoderdecoder/legal-gpt2-final/model_state.pt


tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'TokenizersBackend'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


{'vocab_size': 50000, 'd_model': 600, 'n_encoder_layers': 8, 'd_ff': 2048, 'heads_selected_from_original_source': 15, 'encoder_position_limit': 1536, 'tokenizer_size': 50000}


## 3. Rebuild the exact Phase-1 encoder (without its decoder)

The code below preserves your original **pre-LayerNorm, MultiheadAttention, residual connections, GELU feed-forward** design and module names. The only forward-pass addition is a **padding mask** for batched NER. The saved encoder parameters are loaded strictly (no ignored/mismatched encoder weights).

In [4]:
# Same encoder block as Phase 1: self-attention, residual connections, then FFN.
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # Normalize before attention, as in the original pre-LN block.
        z = self.norm1(x)
        attention_output, _ = self.self_attention(
            z, z, z, key_padding_mask=key_padding_mask, need_weights=False)
        # Add the attention output back to the input (residual connection).
        x = x + self.dropout(attention_output)
        z = self.norm2(x)
        return x + self.dropout(self.feed_forward(z))

# NER uses token features from the encoder. The old text-generation decoder is not used.
class Phase1A2Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, source_length, n_heads, n_layers, d_ff):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.source_position = nn.Embedding(source_length, d_model)
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])

    def forward(self, input_ids, attention_mask):
        seq_len = input_ids.shape[1]
        assert seq_len <= self.source_position.num_embeddings
        # Add a learned position vector to each token embedding.
        pos = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.source_position(pos)
        # Tell attention to ignore padding positions in the batch.
        padding_mask = (attention_mask == 0)
        for block in self.encoder_blocks:
            x = block(x, key_padding_mask=padding_mask)
        return x

encoder = Phase1A2Encoder(vocab_size, d_model, source_length, n_heads, n_layers, d_ff)
# Copy only encoder weights and make sure none are missing.
expected_keys = set(encoder.state_dict())
encoder_state = {key: val for key, val in original_state.items() if key in expected_keys}
missing = expected_keys - set(encoder_state)
if missing:
    raise RuntimeError(f'Missing encoder weights in Phase-1 checkpoint: {sorted(missing)[:20]}')
# strict=True will fail if an encoder weight has the wrong name or shape.
encoder.load_state_dict(encoder_state, strict=True)
print('Loaded ALL', len(encoder_state), 'Phase-1 encoder tensors with strict=True.')
# Free large unused Phase-1 decoder state in RAM.
del original_state, encoder_state
gc.collect()

Loaded ALL 98 Phase-1 encoder tensors with strict=True.


104

## 4. Download official Phase-2 L-NER data and inspect schema

The capstone specifies `fold1-text`, `fold1-gold`, `fold2-text`, `fold2-gold`, `fold3-text`, and `labels.txt`. This notebook resolves them by filename to tolerate an extra Hugging Face directory prefix. No external corpus is used.

In [5]:
# Find the official Phase 2 files on the course dataset repo.
dataset_files = api.list_repo_files(DATA_REPO, repo_type='dataset')

# Match by filename because the repo keeps files inside phase folders.
def official_dataset_file(basename):
    candidates = [path for path in dataset_files if path.split('/')[-1] == basename
                  and 'phase_2' in path.lower()]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected exactly one Phase-2 {basename}; found {candidates}')
    return hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                           filename=candidates[0], token=HF_TOKEN)

def load_json(basename):
    with open(official_dataset_file(basename), encoding='utf-8') as f:
        return json.load(f)

# Fold 1 and Fold 2 have annotations. Fold 3 is the test set.
raw_f1_text = load_json('fold1-text.json')
raw_f2_text = load_json('fold2-text.json')
raw_f1_gold = load_json('fold1-gold.json')
raw_f2_gold = load_json('fold2-gold.json')
raw_test = load_json('fold3-text.json')
with open(official_dataset_file('labels.txt'), encoding='utf-8') as f:
    label_file_lines = [line.strip() for line in f if line.strip()]

# Keep the 12 label spellings exactly as required by the evaluator.
LABELS = ['APP','RESP','A.COUNSEL','R.COUNSEL','JUDGE','WIT','AUTH',
          'COURT','STAT','PREC','DATE','CASENO']
print('labels.txt:', label_file_lines)
print('Examples (raw schema only):',
      type(raw_f1_text).__name__, type(raw_f1_gold).__name__, type(raw_test).__name__)

fold1-text.json: 0.00B [00:00, ?B/s]

fold2-text.json: 0.00B [00:00, ?B/s]

fold1-gold.json: 0.00B [00:00, ?B/s]

fold2-gold.json: 0.00B [00:00, ?B/s]

fold3-text.json: 0.00B [00:00, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

labels.txt: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Examples (raw schema only): dict dict dict


## 5. Normalize documents/annotations and split training vs validation by document

Keep the original document text **unchanged**: offsets refer to that original string. If the published data uses an unrecognized structure, fail explicitly rather than inventing text or silently changing offsets.

In [6]:
# Convert the different possible JSON layouts into {document_id: original_text}.
def normalize_documents(raw):
    if isinstance(raw, dict):
        items = raw.items()
    elif isinstance(raw, list):
        items = [(x.get('id', x.get('doc_id', x.get('document_id'))), x) for x in raw]
    else:
        raise TypeError('Unexpected document JSON shape')
    out = {}
    # Do not lowercase, strip or edit the text; all gold offsets use this exact string.
    for doc_id, rec in items:
        if isinstance(rec, str):
            document = rec
        elif isinstance(rec, dict):
            document = rec.get('text', rec.get('document'))
        else:
            raise TypeError(f'Unknown document entry {doc_id}: {type(rec)}')
        if not isinstance(document, str):
            raise TypeError(f'{doc_id} text is not a single raw string; must preserve exact offsets.')
        out[str(doc_id)] = document
    return out

# Convert gold entities to (start, end, label, mention) tuples.
def normalize_gold(raw):
    if isinstance(raw, dict):
        items = raw.items()
    elif isinstance(raw, list):
        items = [(x.get('id', x.get('doc_id', x.get('document_id'))), x) for x in raw]
    else:
        raise TypeError(f'Unknown gold JSON shape: {type(raw)}')
    answer = {}
    for doc_id, rec in items:
        if isinstance(rec, dict):
            rec = next((rec[key] for key in ['entities','annotations','spans','labels'] if key in rec), None)
        if not isinstance(rec, list):
            raise TypeError(f'Unknown gold format for {doc_id}: {type(rec)}')
        spans = []
        for entry in rec:
            if isinstance(entry, (list, tuple)):
                start, end, label = entry[:3]
                mention = entry[3] if len(entry) > 3 else None
            elif isinstance(entry, dict):
                start = entry.get('start', entry.get('start_offset'))
                end = entry.get('end', entry.get('end_offset'))
                label = entry.get('label', entry.get('entity_type', entry.get('type')))
                mention = entry.get('entity_text')
            else:
                raise TypeError(f'Unknown entity entry type for {doc_id}')
            # Catch any label that is not part of the official 12.
            if label not in LABELS:
                raise ValueError(f'Unknown label {label} in {doc_id}')
            spans.append((int(start), int(end), label, mention))
        answer[str(doc_id)] = sorted(spans, key=lambda x:(x[0],x[1]))
    return answer

# Merge annotated folds only after their formats have been checked.
f1 = normalize_documents(raw_f1_text)
f2 = normalize_documents(raw_f2_text)
g1 = normalize_gold(raw_f1_gold)
g2 = normalize_gold(raw_f2_gold)
test_docs = normalize_documents(raw_test)

if set(f1) & set(f2):
    raise ValueError('Fold 1 and Fold 2 IDs overlap; cannot merge safely.')
all_docs = {**f1, **f2}
all_gold = {**g1, **g2}
if not set(all_docs).issubset(all_gold):
    raise ValueError(f'Missing gold labels for {len(set(all_docs)-set(all_gold))} documents')

# Check that each annotated span points to the correct place in the raw text.
for doc_id, spans in all_gold.items():
    if doc_id not in all_docs: continue
    txt = all_docs[doc_id]
    for s,e,label,mention in spans:
        if not (0 <= s < e <= len(txt)):
            raise ValueError(f'Invalid offsets in {doc_id}: {(s,e,label)} len={len(txt)}')
        if mention is not None and isinstance(mention, str) and txt[s:e] != mention:
            raise ValueError(f'Gold offset/text mismatch in {doc_id}: {txt[s:e]!r} != {mention!r}')

# Split by document, not by chunk, so a case cannot appear in both train and val.
ids = list(all_docs)
random.Random(SEED).shuffle(ids)
n_val = max(1, int(len(ids)*VAL_FRACTION))
val_ids = set(ids[:n_val]); train_ids = set(ids[n_val:])
train_docs = {k:v for k,v in all_docs.items() if k in train_ids}
val_docs = {k:v for k,v in all_docs.items() if k in val_ids}
train_gold = {k:all_gold[k] for k in train_docs}
val_gold = {k:all_gold[k] for k in val_docs}
print('Train docs:',len(train_docs),'Validation docs:',len(val_docs),'Test docs:',len(test_docs))
print('Train entity counts:', dict(sorted(__import__('collections').Counter(
      s[2] for spans in train_gold.values() for s in spans).items())))
# These limits are only for quick debugging runs. Leave them as None for training.
if MAX_TRAIN_DOCS is not None:
    train_docs = dict(list(train_docs.items())[:MAX_TRAIN_DOCS])
if MAX_VAL_DOCS is not None:
    val_docs = dict(list(val_docs.items())[:MAX_VAL_DOCS])
train_gold = {k:all_gold[k] for k in train_docs}
val_gold = {k:all_gold[k] for k in val_docs}
assert train_docs and val_docs


Train docs: 63 Validation docs: 7 Test docs: 35
Train entity counts: {'A.COUNSEL': 180, 'APP': 346, 'AUTH': 930, 'CASENO': 703, 'COURT': 846, 'DATE': 1412, 'JUDGE': 218, 'PREC': 1050, 'R.COUNSEL': 167, 'RESP': 337, 'STAT': 1178, 'WIT': 174}


## 6. Token windows and BIO labels

First label the full tokenized document, then cut it into overlapping windows. That way a window starting in the middle of an entity does not invent a new B-tag.

In [7]:
# One outside tag, then a B and I tag for every entity type.
TAGS = ['O'] + [prefix + '-' + lab for lab in LABELS for prefix in ('B', 'I')]
label_to_id = {tag: i for i, tag in enumerate(TAGS)}
id_to_label = {i: tag for tag, i in label_to_id.items()}
PAD = tokenizer.pad_token_id
assert PAD is not None
assert 1 <= WINDOW_STEP <= MAX_TOKENS


def make_tags(offsets, spans):
    """Assign BIO tags to the whole document before we make windows."""
    tags = [label_to_id['O']] * len(offsets)
    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    ptr = 0
    prev_span = None

    for i, (a, b) in enumerate(offsets):
        if a >= b:
            continue
        # Skip entities that end before this token.
        while ptr < len(spans) and spans[ptr][1] <= a:
            ptr += 1
        if ptr == len(spans):
            break
        s, e, label, _ = spans[ptr]
        # BPE tokens sometimes include the space before an entity.
        # Overlap works for those; exact containment misses them.
        if a < e and b > s:
            if prev_span != ptr:
                tags[i] = label_to_id['B-' + label]
            else:
                tags[i] = label_to_id['I-' + label]
            prev_span = ptr
        else:
            prev_span = None
    return tags


def full_tokens(text):
    # No lowercasing or stripping: the original character offsets must stay intact.
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True,
                        truncation=False)
    ids = encoded['input_ids']
    offsets = [tuple(int(x) for x in pair) for pair in encoded['offset_mapping']]
    assert len(ids) == len(offsets)
    return ids, offsets


def encode_document(text, spans=None):
    ids, offsets = full_tokens(text)
    labels = make_tags(offsets, spans) if spans is not None else None
    windows = []
    for start in range(0, len(ids), WINDOW_STEP):
        end = min(start + MAX_TOKENS, len(ids))
        w = {'ids': ids[start:end], 'offsets': offsets[start:end]}
        if labels is not None:
            w['labels'] = labels[start:end]
        windows.append(w)
        if end == len(ids):
            break
    return windows


class NERWindows(Dataset):
    def __init__(self, docs, gold, training=False):
        self.windows = []
        for doc_id, text in tqdm(docs.items(), desc='Making windows', leave=False):
            spans = gold[doc_id] if gold is not None else None
            windows = encode_document(text, spans)
            if training and MAX_TRAIN_WINDOWS_PER_DOC is not None and len(windows) > MAX_TRAIN_WINDOWS_PER_DOC:
                positive = [i for i, w in enumerate(windows) if any(x != 0 for x in w['labels'])]
                chosen = positive[:MAX_TRAIN_WINDOWS_PER_DOC]
                remaining = [i for i in range(len(windows)) if i not in chosen]
                rng = random.Random(SEED + sum(map(ord, str(doc_id))))
                chosen += rng.sample(remaining, min(len(remaining), max(0, MAX_TRAIN_WINDOWS_PER_DOC-len(chosen))))
                windows = [windows[i] for i in sorted(chosen)]
            for w in windows:
                w['doc_id'] = doc_id
                self.windows.append(w)

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        return self.windows[i]


def collate(batch):
    length = max(len(w['ids']) for w in batch)
    x = torch.full((len(batch), length), PAD, dtype=torch.long)
    mask = torch.zeros_like(x)
    tags = torch.full_like(x, -100)
    for i, w in enumerate(batch):
        n = len(w['ids'])
        x[i, :n] = torch.tensor(w['ids'])
        mask[i, :n] = 1
        if 'labels' in w:
            tags[i, :n] = torch.tensor(w['labels'])
    return {'input_ids': x, 'attention_mask': mask, 'labels': tags}


# Validation/test windows are never sampled or dropped.
train_data = NERWindows(train_docs, train_gold, training=True)
val_data = NERWindows(val_docs, val_gold, training=False)
assert len(train_data) and len(val_data), 'The tokenizer produced no training/validation tokens.'
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate)
print('Train windows:', len(train_data), '| Val windows:', len(val_data))
print('BIO labels:', len(TAGS), '| Window size:', MAX_TOKENS, '| Step:', WINDOW_STEP)


Making windows:   0%|          | 0/63 [00:00<?, ?it/s]

Making windows:   0%|          | 0/7 [00:00<?, ?it/s]

Train windows: 2132 | Val windows: 246
BIO labels: 25 | Window size: 256 | Step: 192


## 7. Check the BIO alignment before training

This catches the earlier problem where the label conversion only recovered a few gold entities. It reports exact reconstruction, not the model score.

In [8]:
def bio_to_spans(text, offsets, tag_ids):
    """Convert token BIO tags back to [start, end, label] character spans."""
    spans = []
    current = None

    def finish():
        nonlocal current
        if current is None:
            return
        s, e, label = current
        # Offsets from the BPE tokenizer can include surrounding whitespace.
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        if s < e:
            spans.append((s, e, label))
        current = None

    for (a, b), idx in zip(offsets, tag_ids):
        if not (0 <= a < b <= len(text)):
            finish()
            continue
        tag = id_to_label[int(idx)]
        if tag == 'O':
            finish()
            continue
        kind, label = tag.split('-', 1)
        # An I-tag without an earlier B is treated as a new span.
        if kind == 'I' and current is not None and current[2] == label:
            current = (current[0], max(current[1], b), label)
        else:
            finish()
            current = (a, b, label)
    finish()
    return sorted(set(spans))


def audit_alignment(docs, gold, limit=None):
    total = 0
    matched = 0
    misses = []
    for doc_id, text in list(docs.items())[:limit]:
        ids, offsets = full_tokens(text)
        gold_spans = gold[doc_id]
        built_tags = make_tags(offsets, gold_spans)
        back = set(bio_to_spans(text, offsets, built_tags))
        expected = {(s, e, lab) for s, e, lab, _ in gold_spans}
        total += len(expected)
        matched += len(expected & back)
        for span in sorted(expected - back)[:5]:
            if len(misses) < 20:
                misses.append({'document': doc_id, 'span': span,
                               'text': text[span[0]:span[1]]})
    rate = matched / max(1, total)
    return {'gold': total, 'recovered': matched, 'recovery': rate,
            'sample_missed_spans': misses}

alignment = {
    'train': audit_alignment(train_docs, train_gold),
    'validation': audit_alignment(val_docs, val_gold)
}
with open(OUT / 'alignment_audit.json', 'w', encoding='utf-8') as f:
    json.dump(alignment, f, indent=2, ensure_ascii=False)
print('Gold span round-trip:', {k: {x: v[x] for x in ('gold', 'recovered', 'recovery')}
                                  for k, v in alignment.items()})
print('Examples missed:', alignment['validation']['sample_missed_spans'][:3])
if STOP_IF_ALIGNMENT_BAD:
    assert alignment['train']['recovery'] >= MIN_GOLD_RECOVERY, 'Token/gold alignment is too low: fix this before training.'
    assert alignment['validation']['recovery'] >= MIN_GOLD_RECOVERY, 'Validation gold-span reconstruction is too low.'


Gold span round-trip: {'train': {'gold': 7541, 'recovered': 7405, 'recovery': 0.9819652565972683}, 'validation': {'gold': 852, 'recovered': 836, 'recovery': 0.9812206572769953}}
Examples missed: [{'document': '975074', 'span': (7995, 8085, 'PREC'), 'text': 'Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]'}, {'document': '19436407', 'span': (9164, 9216, 'PREC'), 'text': 'Merlin Projects Ltd. v. Sri Amar Chand Agarwal &Ors.'}, {'document': '19436407', 'span': (14970, 14999, 'PREC'), 'text': 'Jagdamba Commercial Pvt. Ltd.'}]


## 8. Add the NER head; only fine-tune the last encoder blocks

In [9]:
# One output score per BIO tag for every input token.
class LegalNER(nn.Module):
    def __init__(self, encoder, d_model, num_tags):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(d_model, num_tags)

    def forward(self, input_ids, attention_mask):
        # Shape goes from [batch, tokens] to [batch, tokens, hidden_size].
        features = self.encoder(input_ids, attention_mask)
        return self.classifier(self.dropout(features))

# Keep the pretrained encoder and initialize only the small classifier head.
model = LegalNER(encoder,len(encoder.token_embedding.weight[0]),len(TAGS))
# Freeze the encoder first, then unfreeze its last few blocks.
for p in model.encoder.parameters(): p.requires_grad = False
for block in list(model.encoder.encoder_blocks)[-UNFREEZE_LAST_N:]:
    for p in block.parameters(): p.requires_grad = True
# The new NER classification layer always needs training.
for p in model.classifier.parameters(): p.requires_grad = True
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,} / {sum(p.numel() for p in model.parameters()):,}')
assert trainable > 0

Trainable parameters: 7,825,121 / 62,177,009


## 9. Strict-span decoding and validation

All logits for the same token are averaged across windows before choosing BIO tags. Viterbi rules out invalid I-tag transitions. Validation is done at the full-document character-span level.

In [10]:
def decode_viterbi(logits):
    """Choose the best BIO sequence without illegal I-tags."""
    scores = np.asarray(logits, dtype=np.float32)
    if not len(scores):
        return []
    n_tags = len(TAGS)
    dp = scores[0].copy()
    for j, tag in enumerate(TAGS):
        if tag.startswith('I-'):
            dp[j] = -1e20
    back = np.zeros((len(scores), n_tags), dtype=np.int16)

    # The I-label is allowed only after B/I of the same entity class.
    # O and B can follow any tag, so they share the best previous state.
    for t in range(1, len(scores)):
        next_dp = np.empty_like(dp)
        best_prev = int(dp.argmax())
        for j, tag in enumerate(TAGS):
            if tag.startswith('I-'):
                label = tag[2:]
                b = label_to_id['B-' + label]
                i = label_to_id['I-' + label]
                prev = b if dp[b] >= dp[i] else i
            else:
                prev = best_prev
            back[t, j] = prev
            next_dp[j] = scores[t, j] + dp[prev]
        dp = next_dp

    path = [int(dp.argmax())]
    for t in range(len(scores)-1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]


@torch.inference_mode()
def predict_documents(model, docs, method='viterbi'):
    model.eval()
    output = {'viterbi': {}, 'greedy': {}} if method == 'both' else {}
    for doc_id, text in tqdm(docs.items(), desc='NER predictions', leave=False):
        windows = encode_document(text)
        # A key is a token's start/end char offset in the *original* document.
        # We store logits, not separately decoded spans from each window.
        summed = {}
        counts = Counter()
        for w in windows:
            x = torch.tensor([w['ids']], dtype=torch.long, device=DEVICE)
            mask = torch.ones_like(x)
            with torch.autocast(device_type='cuda', dtype=torch.float16,
                                enabled=(DEVICE.type == 'cuda')):
                scores = model(x, mask)[0].float().cpu().numpy()
            for i, (a, b) in enumerate(w['offsets']):
                if not (0 <= a < b <= len(text)):
                    continue
                key = (a, b)
                if key not in summed:
                    summed[key] = scores[i].copy()
                else:
                    summed[key] += scores[i]
                counts[key] += 1

        offsets = sorted(summed)
        if not offsets:
            if method == 'both':
                output['viterbi'][str(doc_id)] = []
                output['greedy'][str(doc_id)] = []
            else:
                output[str(doc_id)] = []
            continue
        averaged = np.stack([summed[k] / counts[k] for k in offsets])
        if method == 'both':
            for decoder in ('viterbi', 'greedy'):
                tags = (decode_viterbi(averaged) if decoder == 'viterbi'
                        else averaged.argmax(axis=-1).tolist())
                output[decoder][str(doc_id)] = [[s, e, lab] for s, e, lab in bio_to_spans(text, offsets, tags)]
        else:
            if method == 'viterbi':
                tags = decode_viterbi(averaged)
            elif method == 'greedy':
                tags = averaged.argmax(axis=-1).tolist()
            else:
                raise ValueError('Unknown decoder: ' + str(method))
            output[str(doc_id)] = [[s, e, lab] for s, e, lab in bio_to_spans(text, offsets, tags)]
    return output


def strict_macro_f1(pred, gold):
    per_class = {}
    for label in LABELS:
        tp = fp = fn = 0
        for doc_id, true_spans in gold.items():
            truth = {(s, e, lab) for s, e, lab, _ in true_spans if lab == label}
            guessed = {(s, e, lab) for s, e, lab in pred.get(doc_id, []) if lab == label}
            tp += len(truth & guessed)
            fp += len(guessed - truth)
            fn += len(truth - guessed)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[label] = {'precision': precision, 'recall': recall, 'f1': f1,
                            'tp': tp, 'fp': fp, 'fn': fn}
    return sum(d['f1'] for d in per_class.values()) / len(LABELS), per_class


## 10. Train and keep the best validation checkpoint

The model does **not** stop because training loss fails to decrease. Validation F1 decides which checkpoint to keep. Set `USE_EARLY_STOPPING=True` only if I specifically want the optional patience-5 rule.

In [11]:
# The encoder is pretrained; the classifier is new and uses a higher learning rate.
optimizer = torch.optim.AdamW([
    {'params': [p for p in model.encoder.parameters() if p.requires_grad],
     'lr': LR_ENCODER},
    {'params': list(model.classifier.parameters()), 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)
class_weights = torch.ones(len(TAGS), device=DEVICE)
class_weights[label_to_id['O']] = O_WEIGHT
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
BEST_FILE = OUT / 'best_phase2_a2_ner.pt'

best_f1 = -1.0
best_epoch = None
history = []
no_improve = 0
train_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    steps = 0
    epoch_start = time.perf_counter()
    bar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for step, batch in enumerate(bar, start=1):
        x = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        y = batch['labels'].to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16,
                            enabled=(DEVICE.type == 'cuda')):
            logits = model(x, mask)
            raw_loss = criterion(logits.reshape(-1, len(TAGS)), y.reshape(-1))
            loss = raw_loss / ACCUM_STEPS
        scaler.scale(loss).backward()
        running_loss += float(raw_loss.item())
        steps = step
        # One optimizer step after ACCUM_STEPS mini-batches, or at the last batch.
        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        if step % 20 == 0:
            bar.set_postfix(loss=f'{running_loss / step:.4f}')

    # Use complete held-out documents and strict exact spans for validation.
    # Calculate both decoders using the same forward pass over validation documents.
    val_results = predict_documents(model, val_docs, method='both')
    val_viterbi = val_results['viterbi']
    val_greedy = val_results['greedy']
    f1_v, detail_v = strict_macro_f1(val_viterbi, val_gold)
    f1_g, detail_g = strict_macro_f1(val_greedy, val_gold)
    selected_method = 'viterbi' if f1_v >= f1_g else 'greedy'
    selected_f1 = max(f1_v, f1_g)
    record = {
        'epoch': epoch,
        'train_loss': running_loss / max(1, steps),
        'val_viterbi_f1': f1_v,
        'val_greedy_f1': f1_g,
        'selected_decoder': selected_method,
        'val_strict_macro_f1': selected_f1,
        'seconds': round(time.perf_counter() - epoch_start, 1),
    }
    history.append(record)
    with open(OUT / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    print(record)

    if selected_f1 > best_f1:
        best_f1 = selected_f1
        best_epoch = epoch
        no_improve = 0
        torch.save({
            'model_state': {k: v.detach().cpu() for k, v in model.state_dict().items()},
            'tags': TAGS,
            'method': 'partial-last-two-encoder-blocks',
            'phase1_repo': PHASE1_REPO,
            'phase1_weight_file': weights_rel,
            'd_model': d_model, 'n_heads': n_heads, 'n_layers': n_layers,
            'source_length': source_length, 'd_ff': d_ff,
            'validation_f1': best_f1, 'epoch': epoch,
            'decoder': selected_method,
        }, BEST_FILE)
        with open(OUT / 'validation_per_label.json', 'w') as f:
            json.dump(detail_v if selected_method == 'viterbi' else detail_g, f, indent=2)
        print('New best checkpoint:', BEST_FILE)
    else:
        no_improve += 1

    # An increasing training loss does NOT stop the run.
    if USE_EARLY_STOPPING and no_improve >= PATIENCE:
        print('Stopping: validation F1 did not improve for', PATIENCE, 'epochs.')
        break

print('Training took', round((time.perf_counter() - train_start) / 60, 1), 'minutes')
print('Best validation F1:', best_f1, 'at epoch', best_epoch)


Epoch 1/30:   0%|          | 0/2132 [00:00<?, ?it/s]

NER predictions:   0%|          | 0/7 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.8459987461140579, 'val_viterbi_f1': 0.04198102892132743, 'val_greedy_f1': 0.027832736637145906, 'selected_decoder': 'viterbi', 'val_strict_macro_f1': 0.04198102892132743, 'seconds': 30.5}
New best checkpoint: /kaggle/working/phase2_a2_partial_updated/best_phase2_a2_ner.pt


Epoch 2/30:   0%|          | 0/2132 [00:00<?, ?it/s]

NER predictions:   0%|          | 0/7 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.5600629376970917, 'val_viterbi_f1': 0.04314595310713576, 'val_greedy_f1': 0.03725098519182905, 'selected_decoder': 'viterbi', 'val_strict_macro_f1': 0.04314595310713576, 'seconds': 28.2}
New best checkpoint: /kaggle/working/phase2_a2_partial_updated/best_phase2_a2_ner.pt


... (56 outputs left out)

{'epoch': 21, 'train_loss': 0.20363488200247418, 'val_viterbi_f1': 0.08597761445801172, 'val_greedy_f1': 0.04689864004205685, 'selected_decoder': 'viterbi', 'val_strict_macro_f1': 0.08597761445801172, 'seconds': 28.5}


Epoch 22/30:   0%|          | 0/2132 [00:00<?, ?it/s]

NER predictions:   0%|          | 0/7 [00:00<?, ?it/s]

{'epoch': 22, 'train_loss': 0.19482791414285625, 'val_viterbi_f1': 0.093923758260686, 'val_greedy_f1': 0.0497957130407346, 'selected_decoder': 'viterbi', 'val_strict_macro_f1': 0.093923758260686, 'seconds': 28.3}
Stopping: validation F1 did not improve for 5 epochs.
Training took 10.5 minutes
Best validation F1: 0.10219527349383234 at epoch 17


## 12. Save Phase 2 artifacts to Hugging Face

By default this uploads the training checkpoint, logs, validation diagnostics, tokenizer, `prediction.json` and `submission.zip` to a **separate private** repo. The cached Phase-1 weights are excluded.

In [12]:
# The best run may be from an earlier epoch, not necessarily the final one.
best = torch.load(BEST_FILE, map_location='cpu', weights_only=True)
assert best['phase1_repo'] == PHASE1_REPO and best['tags'] == TAGS
model.load_state_dict(best['model_state'], strict=True)
model.to(DEVICE).eval()

# Compare the two decoding rules once more on the saved checkpoint.
validation_preds = predict_documents(model, val_docs, method='both')
comparison = {method: strict_macro_f1(validation_preds[method], val_gold)[0]
              for method in ('viterbi', 'greedy')}
DECODER = max(comparison, key=comparison.get) if AUTO_SELECT_DECODER else best['decoder']
with open(OUT / 'decoder_comparison.json', 'w') as f:
    json.dump({'scores': comparison, 'selected': DECODER, 'best_epoch': best['epoch']}, f, indent=2)
print('Saved best epoch:', best['epoch'], 'Decoder:', DECODER, 'Validation:', comparison)

predictions = predict_documents(model, test_docs, method=DECODER)
assert set(predictions) == set(test_docs), 'The submission has missing or extra document IDs.'
for doc_id, entities in predictions.items():
    assert len(entities) == len(set(tuple(x) for x in entities)), 'Duplicate span in ' + doc_id
    for s, e, lab in entities:
        assert type(s) is int and type(e) is int and 0 <= s < e <= len(test_docs[doc_id])
        assert lab in LABELS, (doc_id, lab)

PREDICTION_FILE = OUT / 'prediction.json'
SUBMISSION_ZIP = OUT / 'submission.zip'
with open(PREDICTION_FILE, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False)
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_FILE, arcname='prediction.json')
with zipfile.ZipFile(SUBMISSION_ZIP) as z:
    assert z.namelist() == ['prediction.json']
    assert set(json.loads(z.read('prediction.json'))) == set(test_docs)

# Keep the settings together with the weights. Useful when restarting Kaggle later.
config = {
    'phase1_repo': PHASE1_REPO, 'phase1_weights': weights_rel, 'seed': SEED,
    'architecture': 'a2', 'technique': 'partial', 'checkpoint_epoch': best['epoch'],
    'epochs_requested': EPOCHS, 'early_stopping_enabled': USE_EARLY_STOPPING,
    'patience': PATIENCE, 'max_tokens': MAX_TOKENS, 'window_step': WINDOW_STEP,
    'batch_size': BATCH_SIZE, 'accumulation_steps': ACCUM_STEPS,
    'encoder_learning_rate': LR_ENCODER, 'head_learning_rate': LR_HEAD,
    'o_weight': O_WEIGHT, 'best_validation_f1': best_f1,
    'final_decoder': DECODER, 'unfreeze_last_n': UNFREEZE_LAST_N,
    'source_length': source_length, 'hidden_size': d_model, 'num_heads': n_heads,
}
(OUT / 'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print('Test documents:', len(predictions))
print('Predicted entities:', sum(len(x) for x in predictions.values()))
print('Files:', PREDICTION_FILE, SUBMISSION_ZIP)


NER predictions:   0%|          | 0/7 [00:00<?, ?it/s]

Saved best epoch: 17 Decoder: viterbi Validation: {'viterbi': 0.10219527349383234, 'greedy': 0.0542729973742776}


NER predictions:   0%|          | 0/35 [00:00<?, ?it/s]

Test documents: 35
Predicted entities: 4346
Files: /kaggle/working/phase2_a2_partial_updated/prediction.json /kaggle/working/phase2_a2_partial_updated/submission.zip


## 12. Optional Hugging Face backup

Upload *Phase 2 outputs* after generating the submission. Phase-1 weights are downloaded but are not uploaded again.

In [13]:
# Save the tokenizer too so it is easy to load the model for later phases.
for filename in ('tokenizer.json', 'tokenizer_config.json'):
    src = TOKENIZER_DIR / filename
    if src.is_file():
        shutil.copy2(src, OUT / filename)

if UPLOAD_PHASE2_CHECKPOINT:
    repo_id = f"{api.whoami()['name']}/Phase2-A2-PartialNER"
    api.create_repo(repo_id, repo_type='model', private=True, exist_ok=True)
    files = [str(p.relative_to(OUT)).replace('\\', '/') for p in OUT.rglob('*')
             if p.is_file() and not p.is_relative_to(CACHE)]
    (OUT / 'upload_manifest.json').write_text(json.dumps(sorted(files), indent=2))
    api.upload_folder(
        repo_id=repo_id, repo_type='model', folder_path=str(OUT),
        token=HF_TOKEN, path_in_repo='',
        ignore_patterns=['phase1_download/**', '**/.ipynb_checkpoints/**', '**/__pycache__/**'],
        commit_message='Save A2 partial NER checkpoint, metrics, diagnostics and submission',
    )
    print('Uploaded to https://huggingface.co/' + repo_id)
else:
    print('HF backup disabled. Files remain in:', OUT)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded to https://huggingface.co/P-261110044/Phase2-A2-PartialNER


### Points to remember for the viva

- **Partial fine-tuning:** The old A2 encoder is loaded strictly; only its last two Transformer blocks and a new BIO classifier are trained. The Phase-1 decoder is not needed.
- **BIO alignment:** Use overlapping character ranges because a BPE token can include a space before an entity. Check how many gold spans can be rebuilt from the labels before training.
- **Overlapping windows:** Tokens appear in more than one window. Average their logits by character offset, then decode the complete document once.
- **Viterbi:** An `I-COURT` tag cannot start the sequence or follow `B-DATE`; it must follow `B-COURT` or `I-COURT`.
- **Loss and epochs:** Cross-entropy is trained with less weight for `O`. A flat training loss does not automatically stop the run. The best epoch is chosen using exact-span validation F1.